# Projekt: Kundenstimmung auf Twitter verstehen

## „Kundensupport auf Twitter“ untersuchen
- mehrstufigen KI-Workflow nutzen, um die Daten eingehend zu verstehen und zu analysieren:
- mit llm , um den Datensatz automatisch zu erkunden und zusammenzufassen.
- mit AutoViz fort , um die Daten visuell zu verstehen.
- mit Hugging-Face-Modell verwendet , um die Stimmung in den Tweets der Kunden zu analysieren.

## Datensatz: Kundensupport auf Twitter

### Dieser Datensatz bietet drei wesentliche Vorteile gegenüber anderen Konversationsdatensätzen:
- Fokussiert : Die Gespräche drehen sich um reale Probleme, die die Menschen gelöst haben möchten – verlorenes Gepäck, Abrechnungsprobleme, stornierte Flüge – wodurch die Daten einen klaren Zweck und eine klare Struktur erhalten.
- Natürlich : Die Sprache ist modern und wirkt authentisch, geschrieben von Menschen mit unterschiedlichem Hintergrund. Sie spiegelt wider, wie Kunden heute tatsächlich online kommunizieren.
- Kurz und bündig : Da Tweets kurz sind, wirken die Antworten authentischer und weniger einstudiert. Dies hilft Modellen, natürlicher zu lernen und unterstützt zudem eine effiziente Verarbeitung.

1. Technische Artefakte.
- Von LLM generierte Datensatzzusammenfassungen oder Explorationsergebnisse
- AutoViz-Visualisierungen, die wichtige Muster hervorheben
- Ergebnisse der Stimmungsanalyse, die mit einem Hugging-Face-Modell erzeugt wurden
2. Analytisches Denken
- Wie Zwillinge Ihr anfängliches Verständnis und Ihre analytische Ausrichtung geprägt haben
- Was AutoViz aufdeckte, war aus den Textzusammenfassungen allein nicht ersichtlich.
- Wie die Ergebnisse der Stimmungsanalyse frühere Erkenntnisse ergänzten oder in Frage stellten
3. Überlegungen zur KI-gestützten Analyse
- Wo KI-Tools die Exploration beschleunigten oder den manuellen Aufwand reduzierten
- Wo menschliche Interpretation noch unerlässlich war
- Stärken und Schwächen der Verwendung von LLMs für die Stimmungsanalyse

Schriftliche Erläuterungen sollten als Markdown-Zellen neben den Ausgaben eingefügt werden.
Ziel ist es, Einsicht, Urteilsvermögen und den effektiven Einsatz von Werkzeugen zu demonstrieren, nicht eine erschöpfende Analyse.

# Ordnerstruktur

In [1]:
# === Core imports & tools ===
import os
import sys
import time
import gc
import re
import json
import threading
import subprocess
import multiprocessing
from pathlib import Path
import importlib.util
import difflib

# === Data & analysis ===
import pandas as pd
import networkx as nx
import psutil
import nbformat
import spacy
import yaml
import requests

# === Global LLM model selection (Single Source of Truth) ===
GLOBAL_MODEL = "deepseek-coder-v2"

In [2]:
# === Infrastruktur & Pfad-Management ===
# Import der in Ordnern enthaltenen .py-Dateien -> darin sind alle anderen Importe & defs

# 1. BASE_DIR: Dynamische Ermittlung des Arbeitsverzeichnisses
# stellt sicher, dass das System auch nach einem Neustart oder Pfadwechsel alles findet.
BASE_DIR = os.getcwd()

# 2. PROJECT_PATHS: Zentrale Mapping-Tabelle (Skelett)
# steuert alle Ein- und Ausgabekanäle; bereits vorhandene Ordner werden nicht überschrieben.
PROJECT_PATHS = {
    # Registry (Meta-Wissen & Agenten)
    "REGISTRY_AGENT":  os.path.join(BASE_DIR, "registry", "agent"),   # YAML-Intelligence
    "REGISTRY_TOOLS":  os.path.join(BASE_DIR, "registry", "tool"),    # JSON-Intelligence
    "REGISTRY_LOGIC":  os.path.join(BASE_DIR, "registry", "logic"),   # Fixe Abläufe & Regeln
    "REGISTRY_SCHEMA": os.path.join(BASE_DIR, "registry", "schema"),  # Spalten-Definitionen (fix)
    "REGISTRY_MAPPING":os.path.join(BASE_DIR, "registry", "mapping"),

    # Output
    "DIST_CODE":       os.path.join(BASE_DIR, "output", "scripts"),   # Generierte Muster
    "DIST_TEXT":       os.path.join(BASE_DIR, "output", "reports"),   # Narrative Analysen
    "DIST_VIS":        os.path.join(BASE_DIR, "output", "visuals"),   # Interaktive Plots

    # Datenquelle
    "DATA_SOURCE":     os.path.join(BASE_DIR, "data")                 # Rohdaten
}

# 3. BEHAVIOR_DIR: Geschützter Ort der Agenten-Rezepte (YAML-Profile)
BEHAVIOR_DIR = PROJECT_PATHS["REGISTRY_AGENT"]


def initialize_global_folders() -> None:
    """
    Erstellt die gesamte Projekt-Struktur.
    Stellt sicher, dass Pfade für Berichte, Skripte, Visuals und Registry existieren,
    bevor das System darauf zugreift.
    """
    for name, folder_path in PROJECT_PATHS.items():
        if not os.path.exists(folder_path):
            os.makedirs(folder_path, exist_ok=True)


# Initialer Infrastruktur-Setup
initialize_global_folders()

In [3]:
# === Daten-Infrastruktur & RAM-Load ===

# 1. PROJEKT-ROOT ermitteln (robust, falls Notebook tiefer im Projekt liegt)
current = Path.cwd()
while not (current / "data").exists() and current.parent != current:
    current = current.parent

DATA_DIR = current / "data"


def initialize_data_infrastructure(live_mode: bool = False) -> None:
    """
    Lädt alle CSV-Dateien aus DATA_DIR in den RAM.
    Ziel: Schneller Zugriff für das Offline-LLM ohne Festplatten-Umwege.
    live_mode ist aktuell ein Platzhalter für spätere Streaming-Varianten.
    """
    file_list = list(DATA_DIR.glob("*.csv"))

    if not file_list:
        print("❌ Keine Datenquelle im 'data' Ordner gefunden.")
        return

    for file_path in file_list:
        # Dateiname für die Variable vorbereiten (z.B. df_sample)
        raw_name = file_path.stem.replace("-", "_").replace(" ", "_")
        var_name = f"df_{raw_name}"

        print(f"💾 Lade {file_path.name} in den RAM...")
        df_tmp = pd.read_csv(
            file_path,
            sep=",",
            engine="c",
            low_memory=False,
            encoding="utf-8-sig"
        )
        # Spaltennamen säubern (keine Leerzeichen/Punkte)
        df_tmp.columns = [
            str(c).strip().replace(" ", "_").replace(".", "")
            for c in df_tmp.columns
        ]
        # Global verfügbar machen
        globals()[var_name] = df_tmp
        print(f"✅ '{var_name}' erstellt. ({len(df_tmp)} Zeilen)")
    gc.collect()
# Initialer Daten-Import (RAM-Load)
initialize_data_infrastructure(live_mode=False)

💾 Lade sample.csv in den RAM...
✅ 'df_sample' erstellt. (93 Zeilen)


In [4]:
# LLM Basis-Einstellungen (Setup).
# Zentrale Modellwahl (Single Source of Truth)
GLOBAL_MODEL = "deepseek-coder-v2"
def __LLM_GLOBAL__(model_name: str | None = None) -> dict:
    """
    Zuständigkeit: Zentrale Basis-Einstellungen (Setup).
    - Identität: Fixierung des Modells.
    - Version: Vorgabe, welche LLM-Setup-Version aktiv ist.
    - Hardware: Feste Ressourcen-Zuweisung für stabilen System-Call.
    """
    # Falls nichts übergeben wurde, immer GLOBAL_MODEL verwenden
    effective_model = model_name or GLOBAL_MODEL
    reserved_threads = max(1, multiprocessing.cpu_count() - 2)
    tools_status = check_available_tools()  # nutzt PROJECT_PATHS intern
    version = "v6"
    config = {
        "temperature": 0.0,                  # Absolute Fakten-Treue
        "num_thread": reserved_threads,      # 1–2 Kerne Reserve
        "timeout": 1200,                     # Ausfallsicherheit für große Pakete
        "num_ctx": 8192,                     # Erweitertes Gedächtnis für Live-Daten/Tabellen
        "repeat_penalty": 1.1,               # Verhindert Logik-Schleifen/Wiederholungen
        "seed": 42,                          # Reproduzierbare Ergebnisse
        "top_p": 0.9,                        # Logische Konsistenz-Absicherung
        "context_priority": "user_instruction_first",
        "exit_signal": "### AGENT_PROCESS_COMPLETE ###",
        "instruction_suffix": "BEENDE MIT: ### AGENT_PROCESS_COMPLETE ###",
    }
    # Sicherstellen, dass der Agenten-Ordner existiert
    if not os.path.exists(BEHAVIOR_DIR):
        os.makedirs(BEHAVIOR_DIR, exist_ok=True)
    return {
        "model_name": effective_model,
        "global_model": GLOBAL_MODEL,
        "version": version,
        "behavior_dir": BEHAVIOR_DIR,
        "registry_tools_path": PROJECT_PATHS["REGISTRY_TOOLS"],
        "tools_status": tools_status,
        "config": config,
        "cores_used": reserved_threads,
    }

def pruefe_und_starte_ollama(target_model: str | None = None) -> bool:
    """
    Zuständigkeit: Sicherstellen, dass die LLM-Engine läuft und das Modell bereit ist.
    Nutzt GLOBAL_MODEL als Default, fragt nur bei Fehlern nach Alternativen mit Vorschlägen.
    """
    # GLOBAL_MODEL als Single Source of Truth
    effective_model = target_model or GLOBAL_MODEL
    url = "http://localhost:11434"
    api_tags = f"{url}/api/tags"
    # 1. Sofort-Check: Läuft die Engine schon?
    server_up = False
    try:
        if requests.get(url, timeout=1).status_code == 200:
            server_up = True
    except:
        pass
    # 2. Start-Versuch je nach Plattform (falls offline)
    if not server_up:
        try:
            print(f"🚀 Starte Ollama Engine ({sys.platform})...")
            if sys.platform == "darwin":
                subprocess.Popen(["open", "-a", "Ollama"],
                               stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
            elif sys.platform == "win32":
                cmd = [os.path.expandvars(r"%LocalAppData%\Ollama\ollama app.exe")]
                if not os.path.exists(cmd[0]):
                    cmd = ["ollama", "serve"]
                subprocess.Popen(cmd, shell=True,
                               stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
            # Warte auf Bootup (max 15 Sek)
            for _ in range(10):
                time.sleep(1.5)
                try:
                    if requests.get(url, timeout=1).status_code == 200:
                        server_up = True
                        break
                except:
                    continue
        except Exception as e:
            print(f"⚠️ Startfehler: {e}")
    if not server_up:
        print("\n🛑 Ollama nicht bereit. Bitte manuell starten.")
        return False
    # 3. Modell-Check mit intelligenten Default-Vorschlägen
    try:
        response = requests.get(api_tags, timeout=2)
        if response.status_code == 200:
            models_data = response.json().get('models', [])
            available_models = [m['name'].split(":")[0] for m in models_data]
            # Fall A: Wunsch-Modell ist verfügbar → direkt OK
            if effective_model in available_models:
                print(f"✅ Multitalent-Check: '{effective_model}' ist einsatzbereit.")
                return True
            # Fall B: Modell fehlt → smarte Alternativen + Vorschläge
            print(f"⚠️ Modell '{effective_model}' nicht gefunden.")
            if not available_models:
                print("❌ Keine lokalen Modelle gefunden. Bitte 'ollama pull' nutzen.")
                return False
            # Verfügbare Modelle anzeigen
            print("\n📋 Verfügbare Modelle:")
            for i, name in enumerate(available_models, 1):
                print(f"  {i}. {name}")
            # Intelligenter Default-Vorschlag (1. Deepseek-Variante oder erstes Modell)
            default_suggestion = next((m for m in available_models if "deepseek" in m.lower()), available_models[0])
            default_num = available_models.index(default_suggestion) + 1
            # Input mit Default-Vorschlag im Prompt
            wahl = input(
                f"\n🎯 Wähle (1-{len(available_models)}) [Default: {default_num} '{default_suggestion}'], "
                f"'p' für Pull '{effective_model}': "
            ).strip()
            # Default bei Enter
            if not wahl:
                wahl = str(default_num)
            if wahl.isdigit() and 1 <= int(wahl) <= len(available_models):
                auswahl = available_models[int(wahl)-1]
                print(f"🔄 Wechsle zu: {auswahl} (temporär für diese Session)")
                # GLOBAL_MODEL bleibt unverändert, nur Session-Override
                return True
            elif wahl.lower() == 'p':
                print(f"📥 Starte Download von {effective_model}...")
                subprocess.run(["ollama", "pull", effective_model])
                return True
            else:
                print("❌ Ungültige Eingabe. Verwende Standard-Modell.")
                return True  # Fallback auf erstes verfügbares
    except Exception as e:
        print(f"⚠️ Fehler beim Modell-Abgleich: {e}")
        return False

def execute_api_raw(payload: dict, config: dict) -> str:
    """
    Zuständigkeit: Physische Kommunikation mit dem lokalen LLM-Server.
    - Nutzt: Ollama /generate API-Endpunkt
    - Absicherung: Timeout aus zentraler Config (__LLM_GLOBAL__)
    - Modell: Automatisch aus payload['model'] (GLOBAL_MODEL)
    """
    url = "http://localhost:11434/api/generate"
    try:
        # Timeout aus zentraler Config (aus __LLM_GLOBAL__)
        response = requests.post(
            url,
            json=payload,
            timeout=config.get("timeout", 120)
        )
        if response.status_code == 200:
            result_json = response.json()
            return result_json.get("response", "").strip()
        else:
            # Detaillierter Fehler mit Modell-Info
            model_used = payload.get("model", "unbekannt")
            return f"API_ERROR({model_used}): Status {response.status_code} - {response.text[:100]}"
    except requests.exceptions.Timeout:
        model_used = payload.get("model", GLOBAL_MODEL)
        return f"TIMEOUT({model_used}): LLM hat {config.get('timeout', 120)}s überschritten."
    except requests.exceptions.ConnectionError:
        return "CONNECTION_ERROR: Ollama Server nicht erreichbar. Läuft 'ollama serve'?"
    except Exception as e:
        model_used = payload.get("model", GLOBAL_MODEL)
        return f"API_CALL_ERROR({model_used}): {str(e)}"

def get_process_signals(setup_data: dict, step_number: int = 1, is_final: bool = False) -> dict:
    """
    Zuständigkeit: Taktung der LLM-Prozesse mit einheitlichen Signalen.
    - Nutzt Timeout direkt aus setup_data['config'] (aus __LLM_GLOBAL__)
    - Konsistente Signal-Struktur für alle Agenten-Schritte
    """
    return {
        "start": f"### START_STEP_{step_number:02d} ###",
        "exit": "### AGENT_PROCESS_COMPLETE ###" if is_final else f"### STEP_{step_number:02d}_DONE ###",
        "timeout": setup_data["config"]["timeout"],
        "model": setup_data.get("model_name", GLOBAL_MODEL),  # Tracking
        "step_info": f"Step {step_number}, Final: {is_final}"
    }

def LLM_type(setup_data: dict) -> str:
    """
    Zuständigkeit: Automatisches Auslesen der Setup-Identität.
    - Liest Modell, Version, Threads direkt aus __LLM_GLOBAL__ Setup
    - Standardisierte Format: model_ver_L_threads
    - Live-Modus (L) + Hardware-Drosselung sichtbar
    """
    # Konsistente Schlüssel-Namen aus __LLM_GLOBAL__
    model = setup_data.get("model_name", GLOBAL_MODEL)
    version = setup_data.get("version", "v6")
    threads = setup_data["config"].get("num_thread", 0)
    return f"{model}_{version}_L_T{threads}"

def get_tool_definition_map():
    """
    Zuständigkeit: Zentrale Wissens-Bibliothek (Warenordnung).
    - Liefert die Definition aller überwachten Tools für den Veredeler.
    - Vollständig bereinigt: Keine Dubletten (Klib, SpaCy etc. sind nun eindeutig).
    - Basiert auf dem 'Warenordnung & Verständnis'-Prinzip.
    """
    auto_tools = {
        # BRAIN & SYSTEM (Kern-Infrastruktur)
        'DeepSeek_Engine':    {'pkg': 'ollama',             'phase': 'BRAIN',       'priority': 0},
        'Python_OS':          {'pkg': 'os',                 'phase': 'SYSTEM',      'priority': 0},
        'Pandas_Core':        {'pkg': 'pandas',             'phase': 'DATA',        'priority': 0},
        'System_Stats':       {'pkg': 'psutil',             'phase': 'SYSTEM',      'priority': 0},
        'NetworkX':           {'pkg': 'networkx',           'phase': 'LOGIC_GRAPH', 'priority': 0},
        'LlamaIndex':         {'pkg': 'llama_index',        'phase': 'LOGIC_META',  'priority': 0},
        'AutoGPTQ':           {'pkg': 'auto_gptq',          'phase': 'LLM_OPT',     'priority': 0},
        'Chromadb':           {'pkg': 'chromadb',           'phase': 'VECTOR_DB',   'priority': 0},
        'Nvidia_SMI':         {'pkg': 'pynvml',             'phase': 'HARDWARE',    'priority': 0},
        'GPUtil':             {'pkg': 'gputil',             'phase': 'HARDWARE',    'priority': 0},

        # REINIGUNG & QA (Phase 1)
        'Klib':               {'pkg': 'klib',               'phase': 'CLEAN',       'priority': 1},
        'Great-Expectations': {'pkg': 'great_expectations', 'phase': 'QA',          'priority': 1},
        'Emoji_Logic':        {'pkg': 'emoji',              'phase': 'NLP_PRE',     'priority': 1},

        # EDA & VISUALISIERUNG (Phase 2)
        'AutoViz':            {'pkg': 'autoviz',            'phase': 'EDA',         'priority': 2},
        'ydata_profiling':    {'pkg': 'ydata_profiling',    'phase': 'EDA',         'priority': 2},
        'Sweetviz':           {'pkg': 'sweetviz',           'phase': 'EDA',         'priority': 2},
        'D-Tale':             {'pkg': 'dtale',              'phase': 'EDA',         'priority': 2},
        'Lux':                {'pkg': 'lux',                'phase': 'EDA',         'priority': 2},
        'MLBox_Prep':         {'pkg': 'mlbox',              'phase': 'PREP',        'priority': 2},

        # ML & FORECASTING (Phase 3)
        'H2O_ai':             {'pkg': 'h2o',                'phase': 'ML',          'priority': 3},
        'TPOT':               {'pkg': 'tpot',               'phase': 'ML',          'priority': 3},
        'PyCaret':            {'pkg': 'pycaret',            'phase': 'ML',          'priority': 3},
        'Auto-Sklearn':       {'pkg': 'autosklearn',        'phase': 'ML',          'priority': 3},
        'Prophet':            {'pkg': 'prophet',            'phase': 'FORECAST',    'priority': 3},
        'NeuralProphet':      {'pkg': 'neuralprophet',      'phase': 'FORECAST',    'priority': 3},

        # NLP & TEXT (Twitter-Projekt Fokus)
        'Transformers':       {'pkg': 'transformers',       'phase': 'NLP',         'priority': 3},
        'SpaCy':              {'pkg': 'spacy',              'phase': 'LOGIC_NLP',   'priority': 0}, # Prio 0 für Logik-Verständnis
        'TextBlob':           {'pkg': 'textblob',           'phase': 'NLP',         'priority': 3},
        'NLTK_Core':          {'pkg': 'nltk',               'phase': 'NLP',         'priority': 3},
        'Vader_Sentiment':    {'pkg': 'vaderSentiment',     'phase': 'NLP',         'priority': 3},
        'BerTopic':           {'pkg': 'bertopic',           'phase': 'NLP_TOPIC',   'priority': 3},

        # INTERAKTION & XAI (Phase 4/5)
        'PandasAI':           {'pkg': 'pandasai',           'phase': 'CHAT',        'priority': 4},
        'LangChain':          {'pkg': 'langchain',          'phase': 'AGENT',       'priority': 4},
        'SHAP':               {'pkg': 'shap',               'phase': 'XAI',         'priority': 4},
        'LIME':               {'pkg': 'lime',               'phase': 'XAI',         'priority': 4},
        'Dalex':              {'pkg': 'dalex',              'phase': 'XAI',         'priority': 4},
        'Giskard':            {'pkg': 'giskard',            'phase': 'QA',          'priority': 5}
    }
    return auto_tools

def check_available_tools() -> dict:
    """
    Zuständigkeit: RAM-schonender Audit der Tool-Infrastruktur.
    - RAM-Schutz: Nutzt find_spec statt import (Pakete werden nicht geladen)
    - Effizienz: Speichert NUR aktive Tools (spart LLM-Token)
    - Output: tool_logic_registry.json in PROJECT_PATHS["REGISTRY_TOOLS"]
    """
    # 1. Warenordnung laden
    auto_tools = get_tool_definition_map()
    active_tools_registry = {}
    logic_map = nx.DiGraph()
    for tool_name, info in auto_tools.items():
        pkg = info['pkg']
        is_installed = False
        # A: Spezial-Check für Brain-Engine (Ollama + GLOBAL_MODEL)
        if tool_name == 'DeepSeek_Engine':
            try:
                check = subprocess.run(['ollama', 'list'],
                                     capture_output=True, text=True, timeout=5)
                is_installed = 'deepseek' in check.stdout.lower()
            except:
                is_installed = False
        # B: RAM-schonender Check für alle anderen Pakete
        else:
            if pkg in sys.builtin_module_names or pkg in sys.modules:
                is_installed = True
            else:
                try:
                    is_installed = importlib.util.find_spec(pkg) is not None
                except:
                    is_installed = False
        # C: NUR aktive Tools in Registry (RAM-Platz sparen)
        if is_installed:
            registry_key = tool_name.upper()
            active_tools_registry[registry_key] = {
                'phase': info['phase'],
                'priority': info['priority'],
                'package': pkg,
                'model_context': GLOBAL_MODEL  # LLM-Kontext für Tool-Sequenzierung
            }
            logic_map.add_node(registry_key, phase=info['phase'])
    # 2. Logik-Kern mit GLOBAL_MODEL Integration
    active_tools_registry['__LOGIC_CORE__'] = {
        'intent_engine': f'DeepSeek-Ready({GLOBAL_MODEL})'
                        if 'DEEPSEEK_ENGINE' in active_tools_registry
                        else 'Llama-Fallback',
        'global_model': GLOBAL_MODEL,
        'knowledge_graph': list(logic_map.nodes()),
        'total_active': len(active_tools_registry),
        'total_monitored': len(auto_tools)
    }
    # 3. Wissen sichern (globaler Registry-Pfad)
    registry_path = os.path.join(PROJECT_PATHS["REGISTRY_TOOLS"], "tool_logic_registry.json")
    with open(registry_path, 'w', encoding='utf-8') as f:
        json.dump(active_tools_registry, f, indent=4)
    print(f"✅ Tool-Registry erstellt: {len(active_tools_registry)} aktive Tools")
    return active_tools_registry

def get_system_logic_dossier(tool_status: dict) -> tuple[str, list]:
    """
    Zuständigkeit: Extrahiert aktive Logik-Hierarchie für LLM-Prompts.
    - Trennt physisch installierte Pakete von aktuell geladenen Modulen
    - Formatiert sauberes Dossier für Tool-Sequenzierung
    """
    # 1. Physisch installierte Tools (aus check_available_tools)
    pip_liste = [t for t, info in tool_status.items()
                if isinstance(info, dict) and info.get('phase')]  # 'phase' = installiert
    # 2. Aktiv im RAM (bereits importiert)
    aktive_logik = [info['package'] for t, info in tool_status.items()
                   if isinstance(info, dict) and
                   info.get('package') in sys.modules]
    # 3. GLOBAL_MODEL Kontext hinzufügen
    llm_status = f"LLM: {GLOBAL_MODEL} (aus __LLM_GLOBAL__)"
    # 4. Sauberes Dossier für LLM-Prompts
    dossier = (
        f"🛠️  INSTALLIERTE TOOLS: {', '.join(pip_liste[:8])}{'...' if len(pip_liste) > 8 else ''}\n"
        f"⚡ AKTIVE IM RAM: {', '.join(aktive_logik[:6])}{'...' if len(aktive_logik) > 6 else ''}\n"
        f"🧠 {llm_status}"
    )
    return dossier, aktive_logik

def erstelle_sequenz_aus_registry(sanitized_text: str, active_registry: dict) -> list[dict]:
    """
    Zuständigkeit: Verwandelt bereinigten Text in logische Tool-Abfolge.
    - Nutzt Prioritäten aus tool_logic_registry.json (check_available_tools)
    - Intelligente Matching + Duplikat-Schutz + Vergleichs-Logik
    """
    try:
        if not active_registry:
            return []
        text_low = sanitized_text.lower()
        treffer = []
        # 1. TOOL-SCAN: Abgleich mit Registry (physisch + RAM-aktiv)
        for tool_name, info in active_registry.items():
            if tool_name.startswith("__"):  # Skip interne Keys
                continue
            # Robustes Matching (Name + Package + GLOBAL_MODEL-Kontext)
            pkg_name = info.get('package', '').lower()
            model_context = info.get('model_context', '').lower()
            if (tool_name.lower() in text_low or
                pkg_name in text_low or
                (GLOBAL_MODEL.lower() in text_low and 'deepseek' in tool_name.lower())):
                treffer.append({
                    'name': tool_name,
                    'prio': info.get('priority', 99),
                    'phase': info.get('phase', 'UNKNOWN'),
                    'pkg': pkg_name,
                    'model_context': model_context
                })

        # 2. PRIORISIERUNG: Nach Phasen/Priorität sortieren
        treffer_sortiert = sorted(treffer, key=lambda x: (x['prio'], x['phase']))
        # 3. VERGLEICHS-LOGIK: Automatisch LLM_COMPARE einfügen
        if "vergleich" in text_low or "compare" in text_low:
            if not any(t['name'] == 'LLM_COMPARE' for t in treffer_sortiert):
                treffer_sortiert.append({
                    'name': 'LLM_COMPARE',
                    'prio': 9,
                    'phase': 'VEREDELUNG',
                    'pkg': 'logic',
                    'model_context': GLOBAL_MODEL
                })
        # 4. DUPLIKAT-REINIGUNG + Maximale Sequenzlänge
        final_queue = []
        seen = set()
        for t in treffer_sortiert:
            if t['name'] not in seen and len(final_queue) < 8:  # Max 8 Tools
                final_queue.append(t)
                seen.add(t['name'])
        return final_queue
    except Exception as e:
        print(f"⚠️ SEQUENZER_FEHLER: {str(e)}")
        return []

def start_spinner(stop_event: threading.Event, is_de: bool = True) -> None:
    """
    VISUELLES FEEDBACK [UX-STANDARD]
    Zeigt Spinner während LLM-Verarbeitung + System-Monitoring.
    """
    def perform_system_check() -> str:
        """
        SYSTEM-CHECK: Validiert RAM & CPU-Last in Echtzeit.
        - Hält 1 Core Reserve frei
        - Trigger GC bei kritischem RAM
        """
        ram_avail = psutil.virtual_memory().available / (1024**3)  # GB
        cpu_usage = psutil.cpu_percent(interval=None)
        status_msg = ""
        # RAM-Überwachung (kritisch < 1.5GB)
        if ram_avail < 1.5:
            gc.collect()
            status_msg = f"RAM↓{ram_avail:.1f}GB|GC"
        # CPU-Überwachung (>90% = Warnung)
        if cpu_usage > 90:
            cores_free = max(1, os.cpu_count() - multiprocessing.cpu_count())
            cpu_alert = f"CPU↑{cpu_usage:.0f}%"
            status_msg = f"{status_msg} | {cpu_alert}" if status_msg else cpu_alert
        return status_msg
    # Spinner-Animation + deutsche/englische Texte
    chars = ['⠋','⠙','⠹','⠸','⠼','⠴','⠦','⠧','⠇','⠏']
    msg = "🤖 LLM denkt... (GPU/CPU)" if is_de else "🤖 LLM thinking... (GPU/CPU)"
    idx = 0
    while not stop_event.is_set():
        alert_msg = perform_system_check()
        display = f'\r{msg} {chars[idx % len(chars)]} {alert_msg}' if alert_msg else f'\r{msg} {chars[idx % len(chars)]}'
        sys.stdout.write(display)
        sys.stdout.flush()
        idx += 1
        time.sleep(1.0)
    # Clean Exit (Cursor zurück)
    sys.stdout.write('\r' + ' ' * 100 + '\r')
    sys.stdout.flush()

def lokalisiere_alle_quellen_final(user_input: str, setup_data: dict) -> list[dict] | None:
    """
    UNIVERSAL-RADAR [v3 - COLUMN-DEEP-SCAN + GLOBAL_MODEL]
    Zuständigkeit: Findet RAM-Objekte über Namen, Spalten & Kontext.
    - Fuzzy-Matching für Tippfehler (tweat → tweet_id)
    - PROJECT_PATHS Integration für Disk-Scan
    """
    ns = globals()
    gefundene_evidenz = []

    # 1. Vorbereitung: Token-Split + DataFrame-Patterns
    input_words = user_input.lower().split()
    input_placeholders = re.findall(r'\b(df\w*|cl\w*|data\w*)\b', user_input.lower())

    # --- 1. RAM & DATAFRAME SCAN ---
    kandidaten = {
        n: o for n, o in ns.items()
        if isinstance(o, pd.DataFrame) or hasattr(o, 'get_schema')
    }

    for name, obj in kandidaten.items():
        name_lower = name.lower()
        korrektur_map = {}
        spalten = list(obj.columns) if hasattr(obj, 'columns') else []

        is_relevant = False
        identifizierte_spalten = []

        # A) DIREKTER NAMEN-MATCH
        for p in input_placeholders:
            if name_lower.startswith(p) or p in name_lower:
                is_relevant = True
                break

        # B) DEEP COLUMN SCAN + FUZZY
        for col in spalten:
            col_l = col.lower()

            # Exakter Match
            if col_l in user_input.lower():
                is_relevant = True
                identifizierte_spalten.append(col)
            # Fuzzy-Match (Tippfehler-Korrektur)
            else:
                match = difflib.get_close_matches(col_l, input_words, n=1, cutoff=0.7)
                if match:
                    is_relevant = True
                    identifizierte_spalten.append(col)
                    korrektur_map[match[0]] = col

        # C) OBJEKT-FUZZY (df_sampl → df_sample)
        if not is_relevant:
            fuzzy_name = difflib.get_close_matches(name_lower, input_words, n=1, cutoff=0.6)
            if fuzzy_name:
                is_relevant = True
                korrektur_map[fuzzy_name[0]] = name

        if is_relevant:
            # Metadaten mit PROJECT_PATHS Kontext
            link = "RAM"
            if hasattr(obj, 'get_origin'):
                link = obj.get_origin()
            elif hasattr(obj, 'attrs') and obj.attrs.get('source'):
                link = obj.attrs['source']

            gefundene_evidenz.append({
                "objekt": name,
                "typ": "DATAFRAME",
                "quelle": link,
                "korrektur_map": korrektur_map,
                "spalten": spalten,
                "identifizierte_spalten": identifizierte_spalten[:5],  # Max 5
                "shape": f"{len(obj)}x{len(obj.columns)}" if hasattr(obj, '__len__') else "unknown",
                "global_model": GLOBAL_MODEL  # LLM-Kontext
            })

    # --- 2. DISK-SCAN (PROJECT_PATHS) ---
    for target_key in ["DIST_CODE", "DIST_TEXT", "DIST_VIS"]:
        if target_key in PROJECT_PATHS:
            pfad = PROJECT_PATHS[target_key]
            if os.path.exists(pfad):
                relevant_files = [
                    f for f in os.listdir(pfad)
                    if any(w in f.lower() for w in input_words)
                ][:5]  # Max 5 Files
                if relevant_files:
                    gefundene_evidenz.append({
                        "objekt": f"DISK_{target_key}",
                        "typ": "OFFLINE_ARCHIVE",
                        "quelle": pfad,
                        "dateien": relevant_files
                    })

    # --- 3. BEGRIFFS-LOGIK (EDA/ML/NLP) ---
    lib_keywords = {
        'plot', 'chart', 'viz', 'autoviz', 'sweetviz',
        'stats', 'eda', 'profile', 'describe',
        'pandasai', 'langchain', 'sentiment', 'nlp'
    }
    libs = [w for w in input_words if w in lib_keywords]
    if libs:
        gefundene_evidenz.append({
            "typ": "LIB_REFERENCE",
            "begriffe": libs,
            "phase": "EDA" if any(k in libs for k in ['plot','viz','autoviz']) else "NLP"
        })
    return gefundene_evidenz if gefundene_evidenz else None

def get_or_create_reg_index(force_refresh: bool = False) -> dict:
    """
    Zuständigkeit: Erstellt Landkarte aller Registry-Dateien (Caching).
    - Lazy Loading: Nur einmal im Speicher als 'REG_INDEX'
    - Nutzt PROJECT_PATHS + BASE_DIR (globalisierte Vorgaben)
    - Best Practice: Singleton-Pattern statt unsauberem global
    """
    # Best-Practice Singleton (kein unsauberer global-Zugriff)
    if 'REG_INDEX' in globals() and not force_refresh:
        return REG_INDEX
    # Base-Pfad aus globalisierten Konstanten
    base_reg = os.path.join(BASE_DIR, "registry")
    reg_index = {}
    # Rekursiver Scan (agent, logic, schema, mapping, etc.)
    if os.path.exists(base_reg):
        for root, dirs, files in os.walk(base_reg):
            for file in files:
                path = Path(os.path.join(root, file))
                reg_index[path.stem.lower()] = {
                    "full_name": file,
                    "path": str(path),
                    "type": path.suffix.lower(),
                    "folder": os.path.basename(root),
                    "global_model": GLOBAL_MODEL,  # LLM-Kontext für Agenten-YAMLs
                    "project_paths": PROJECT_PATHS  # Vollständiger Pfad-Kontext
                }
    # Cache als Konstante (UPPERCASE = immutable convention)
    globals()['REG_INDEX'] = reg_index
    print(f"✅ Registry-Index erstellt: {len(reg_index)} Einträge gefunden.")
    return reg_index

def execute_targeted_registry_task(topic_key: str, task_logic_func) -> any:
    """
    Zuständigkeit: Registry-basierte Task-Execution mit RAM-Management.
    Workflow:
    1. Findet Datei im REG_INDEX (Singleton)
    2. Lädt temporär via registry_universal_loader
    3. Führt Task aus
    4. Reinigt RAM (GC)
    """
    # REG_INDEX Singleton (aus vorheriger Def)
    index = get_or_create_reg_index()
    match = index.get(topic_key.lower())
    if not match:
        return f"❌ Fehler: '{topic_key}' nicht in Registry ({len(index)} Einträge) gefunden."
    print(f"📖 Lade Fachwissen: {match['full_name']} ({match['folder']})...")
    try:
        # Temporäres Laden (Übersetzer-Pattern)
        temp_content = registry_universal_loader(match['path'], match)
        result = task_logic_func(temp_content)
        # RAM-Reinigung (Best Practice)
        del temp_content
        gc.collect()
        print(f"♻️ RAM bereinigt: '{topic_key}' entladen.")
        return result
    except Exception as e:
        print(f"⚠️ TASK_ERROR ({topic_key}): {str(e)}")
        gc.collect()
        return None

def lade_alte_loesung_disk(file_path: str | Path) -> list[dict]:
    """
    Zuständigkeit: Extrahiert Erfolgs-Momente aus alten IPYNB-Dateien.
    - Code-Zellen mit Outputs (Prints/Tables) für LLM-Lernprozess
    - RAM-schonend: Begrenzt Länge, robuste Error-Handling
    """
    if not os.path.exists(file_path):
        return []
    try:
        # Notebook laden (nbformat v4 standard)
        with open(file_path, 'r', encoding='utf-8') as f:
            nb = nbformat.read(f, as_version=4)
        erkenntnisse = []
        for i, cell in enumerate(nb.cells):
            # Nur Code-Zellen mit relevanten Outputs
            if (cell.cell_type == 'code' and
                cell.outputs and
                len(cell.outputs) > 0 and
                cell.source.strip()):
                output_text = ""
                # Priorisiere text/plain, dann stderr, dann alles
                for out in cell.outputs:
                    if out.output_type == 'stream' and out.get('text'):
                        output_text = str(out['text'])[:300]
                        break
                    elif out.output_type == 'display_data' and out.get('data', {}).get('text/plain'):
                        output_text = str(out['data']['text/plain'])[:300]
                        break
                snippet = {
                    "cell_id": i,
                    "code_preview": cell.source[:300].strip(),  # Kurzversion
                    "output": output_text.strip(),
                    "has_table": any('dataframe' in str(out).lower() for out in cell.outputs),
                    "global_model": GLOBAL_MODEL,  # LLM-Kontext
                    "source_file": str(file_path)
                }
                erkenntnisse.append(snippet)
        # Max 20 Erfolgs-Momente (Token-Limit)
        return erkenntnisse[:20]
    except Exception as e:
        print(f"⚠️ IPYNB_PARSE_ERROR ({file_path}): {str(e)}")
        return []

In [5]:
# LLM verhalten lenkung
def apply_system_discipline(language: str = 'DE') -> str:
    """
    Zuständigkeit: Output-Disziplinierung für präzise LLM-Antworten.
    - Hybrid: Code immer EN (Bibliotheken), Info in DE/EN
    - Verhindert Token-Verschwendung durch Smalltalk/Markdown
    - GLOBAL_MODEL-spezifische Anweisungen
    """
    # 1. Sprach-Ebene bestimmen
    lang_info = "DEUTSCH" if language.upper() == 'DE' else "ENGLISH"
    # 2. Verschärftes Protokoll V8 (mit GLOBAL_MODEL)
    instruktion = (
        f"### SYSTEM_DISCIPLINE_V8_HYBRID ###\n"
        f"LLM: {GLOBAL_MODEL} | MODE: RAW-PYTHON-EXECUTABLE\n\n"
        f"LANGUAGES:\n"
        f"• CODE: ENGLISH ONLY (pandas, autoviz, transformers, etc.)\n"
        f"• INFO: {lang_info}\n\n"
        f"STRICT RULES:\n"
        f" START IMMEDIATELY WITH PYTHON CODE (no intro)\n"
        f" NO markdown, backticks(```), headers(###), lists(-)\n"
        f" NO greetings, explanations, 'Here is the code'\n"
        f" NO strategy-talk, analysis, reasoning text\n"
        f" Variable names: df_sample, inbound_message (EN)\n"
        f" END WITH ONE LINE: 'TechInfo: {GLOBAL_MODEL} done.'\n\n"
        f"EXECUTE NOW →"
    )
    return instruktion

def get_agent_intelligence_map(tool_status: dict) -> tuple[dict, str]:
    """
    DELTA-OVERLAY ORCHESTRATOR (Schablonen-Prinzip v2)
    Lehrt LLM: df_name (Original) → cl_name (Korrektur) → v_name (Virtual View)
    - Ohne Originaldaten zu verändern (non-destructive)
    - Phasen-basierte Tool-Zuordnung aus check_available_tools
    """
    # Tools nach Phase gruppieren (aus Registry)
    intelligence_roadmap = {}
    for tool_name, info in tool_status.items():
        if isinstance(info, dict) and info.get('phase'):  # = installiert
            phase = info['phase']
            if phase not in intelligence_roadmap:
                intelligence_roadmap[phase] = []
            intelligence_roadmap[phase].append(tool_name)

    # Kompakte Phasen-Strategie (Token-optimiert)
    phasen_guide = (
        f"DELTA-OVERLAY STRATEGY | LLM: {GLOBAL_MODEL}\n\n"
        f"P0 AUDIT: df_name.scan() → Fehlerliste\n"
        f"P1 SCHABLONE: cl_name = pd.DataFrame(NaN, index=df_name.index)\n"
        f"P2 REPAIR: cl_name.loc[mask, col] = fix_value\n"
        f"P3 VIEW: v_name = cl_name.combine_first(df_name)\n"
        f"P4 ANALYSE: EDA/ML/NLP auf v_name\n"
        f"P5 XAI: shap/lime(df_name vs v_name)\n\n"
        f"RULES: df_name IMMER ORIGINAL | cl_name = Fixes ONLY"
    )
    return intelligence_roadmap, phasen_guide

def build_llm_payload(prompt: str, setup_data: dict) -> dict:
    """
    Zuständigkeit: Baut Ollama-kompatibles Payload aus __LLM_GLOBAL__ Setup.
    - Single Source: Alle Parameter aus setup_data (GLOBAL_MODEL)
    - Stop-Sequenzen: Verhindert unerwünschte Outputs
    """
    # Konsistente Schlüssel aus __LLM_GLOBAL__
    model = setup_data.get("model_name", GLOBAL_MODEL)
    return {
        "model": model,
        "prompt": prompt.strip(),
        "stream": False,  # Batch-Modus für stabile Outputs

        "options": {
            "temperature": setup_data["config"].get("temperature", 0.0),
            "num_ctx": setup_data["config"].get("num_ctx", 8192),
            "num_thread": setup_data["config"].get("num_thread", 4),
            "repeat_penalty": setup_data["config"].get("repeat_penalty", 1.1),
            "seed": setup_data["config"].get("seed", 42),
            "top_p": setup_data["config"].get("top_p", 0.9),
            "stop": [
                "### AGENT_PROCESS_COMPLETE ###",
                "A nice", "Structured", "Hello",
                "Sure", "Gern", "Bitte", "Hier ist",
                "```", "```python", "###", "- ",
                "TechInfo:", "Technical info"
            ]
        }
    }

def apply_radical_cleaning(raw_text: str, exit_signal: str) -> str:
    """
    Stufe 25: Absolute Bereinigung (V6 - Syntax Integrity Edition).
    Zuständigkeit: Isoliert Python-Befehlskern ohne Syntax-Zerstörung.
    - Ersetzt LLM-Disziplin-Verstöße durch reinen Code
    - GLOBAL_MODEL-kompatible Keywords
    """
    if not raw_text or not raw_text.strip():
        return ""
    # 1. Markdown + Discipline-Zäune entfernen (Code bleibt)
    clean = re.sub(r'```python|```|`|###.*###', '', raw_text)
    # 2. FRAGMENT-KILLER (Import-Filter + Discipline-Smalltalk)
    lines = clean.split('\n')
    temp_lines = []
    garbage_patterns = [
        r'^\s*(import|from\s+\w+|as\s+(pd|plt|nx)\b).*',  # Imports
        r'.*(Hier ist|Here is|Strategie|Strategy|TechInfo).*',  # Smalltalk
        r'^\s*-?\s*(START|STEP|AGENT).*',  # Discipline-Tokens
        r'^\s*\d+️⃣?\s*.*'  # Nummerierte Listen
    ]
    for line in lines:
        stripped = line.strip()
        if any(re.search(p, stripped, re.IGNORECASE) for p in garbage_patterns):
            continue
        if stripped and not stripped.startswith('#'):  # Skip Comments
            temp_lines.append(line)
    # 3. INTENT-RECOVERY (Python-Code erkennen)
    python_keywords = [
        'df_', 'dfsample', 'head', 'tail', 'shape', 'info',  # DataFrames
        'plot', 'bar', 'count', 'describe', 'sample',        # EDA
        'inbound', 'tweet', '.head', '.tail'                 # Domain
    ]
    best_line = ""
    for line in temp_lines:
        line_lower = line.lower()
        # Python-Syntax: Funktion-Call oder Method-Chain
        if (any(kw in line_lower for kw in python_keywords) or
            re.search(r'\w+\.\w+\(', line) or
            re.search(r'^\s*[\w\[]+\s*=?\s*', line)):
            best_line = line
            break
    # Fallback: Erste nicht-leere Zeile
    if not best_line and temp_lines:
        best_line = temp_lines[0]
    # 4. RADIKALER CUT (Exit-Signal + Metadaten)
    schneide_begriffe = [
        exit_signal, "TechInfo:", "Technical info", "Datum:", "Note:"
    ]
    for begriff in schneide_begriffe:
        best_line = re.split(re.escape(begriff), best_line, flags=re.IGNORECASE)[0]
    # 5. SYNTAX-POLITUR (Python-konform)
    best_line = best_line.strip()
    best_line = re.sub(r'^[:\-\s•]+', '', best_line)  # Prefix-Cleanup
    # Quote-Entfernung (nur bei vollständig eingeschlossenen Strings)
    if (best_line.startswith(('"', "'")) and best_line.endswith(best_line[0])):
        best_line = best_line[1:-1].strip()
    # Final-Validierung: Mindestlänge + Python-Keywords
    if len(best_line) < 5 or not any(kw in best_line.lower() for kw in ['df', '.', '(', '[', 'plot']):
        return ""
    print(f"🧹 RADICAL-CLEAN V6: '{best_line[:60]}...'")
    return best_line.strip()

def build_veredelungs_prompt(user_input: str, evidenz: list, dossier: str, strenge: str) -> str:
    """
    WORKFLOW-ARCHITEKT V2: User-Slang → Präzise Befehlssequenz.
    Ziel: NULL-PROSA. NUR technischer Befehlskern.
    """
    # 1. Radar-Korrekturen extrahieren (Universal-RADAR)
    korrektur_details = []
    if evidenz:
        for e in evidenz:
            # Tippfehler-Korrekturen (tweat → tweet_id)
            if e.get('korrektur_map'):
                for falsch, richtig in e['korrektur_map'].items():
                    korrektur_details.append(f"KORR: {falsch}→{richtig}")
            # DataFrame + Spalten-Kontext
            if e.get('objekt'):
                spalten = e.get('identifizierte_spalten', [])[:3]
                if spalten:
                    korrektur_details.append(f"DF: {e['objekt']} [{', '.join(spalten)}]")
                else:
                    korrektur_details.append(f"DF: {e['objekt']}")
    hinweis_string = " | ".join(korrektur_details[:4])  # Max 4 Hinweise
    # 2. Zero-Tolerance Prompt (Discipline V8 + GLOBAL_MODEL)
    prompt = (
        f"{strenge}\n"  # apply_system_discipline
        f"LLM: {GLOBAL_MODEL} | ROLLE: TOKEN→CODE\n\n"
        f"REFERENZ: {dossier}\n"
        f"KONEXT: {hinweis_string}\n\n"
        f"STRICT:\n"
        f"• NO prose, NO 'Hier ist', NO explanations\n"
        f"• NO markdown, NO ```, NO lists\n"
        f"• ONLY COMMAND (one line)\n\n"
        f"INPUT: {user_input}\n\n"
        f"OUTPUT:"
        f"Input: 'zeig mir 5 zeilen von df'\n"
        f"Resultat: 'df_sample.head(5)'\n"  # Explizit Pandas Syntax!
        f"Input: 'plot inbound'\n"
        f"Resultat: 'df_sample.inbound.value_counts().plot.bar()'\n"
    )
    return prompt.strip()

# Global NLP-Model Cache (Singleton)
NLP_MODELS = {}


def extrahiere_technischen_kern(text: str, language: str = 'DE') -> str:
    """
    Zuständigkeit: NLP-Precleaning (v4 - Code-Syntax Protection).
    - SpaCy-basierte Keyword-Extraktion mit Python-Schutz
    - Dynamisches Model-Download + GLOBAL_MODEL-Kontext
    """
    if not text or len(text.strip()) < 3:
        return text.strip()
    # 1. Modell-Auswahl (DE/EN)
    model_name = "de_core_news_sm" if language.upper() == 'DE' else "en_core_web_sm"
    # 2. Lazy Loading + Auto-Download
    if model_name not in NLP_MODELS:
        try:
            NLP_MODELS[model_name] = spacy.load(model_name)
        except OSError:
            print(f"📥 Installiere SpaCy-Modell: {model_name}...")
            import subprocess, sys
            subprocess.run([
                sys.executable, "-m", "spacy", "download", model_name
            ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            NLP_MODELS[model_name] = spacy.load(model_name)
    nlp = NLP_MODELS[model_name]
    doc = nlp(text)
    relevante_begriffe = []
    # 3. CODE-AWARE FILTER (Schutz + Technik)
    for token in doc:
        t_low = token.text.lower()
        # A) PYTHON-SYNTAX SCHUTZ (höchste Priorität)
        if re.search(r'[_.\[\]()]', token.text):
            relevante_begriffe.append(token.text)
            continue
        # B) DATA-SCIENCE WHITELIST
        tech_keywords = {
            'df', 'plot', 'count', 'head', 'shape', 'row', 'col',
            'inbound', 'tweet', 'sentiment', 'autoviz', 'groupby'
        }
        if t_low in tech_keywords:
            relevante_begriffe.append(token.text)
            continue
        # C) LINGUISTIK-FILTER (NOUN/VERB + NO STOPWORDS)
        if token.pos_ in ['NOUN', 'VERB', 'PROPN', 'NUM', 'ADJ']:
            if not token.is_stop or t_low in ['wie', 'viel', 'alle', 'erste']:
                relevante_begriffe.append(token.text)
    # 4. SYNTAX-REKONSTRUKTION
    resultat = " ".join(relevante_begriffe)
    # Syntax-Glättung (Code-freundlich)
    syntax_fixes = {
        " . ": ".", " ( ": "(", " ) ": ")", " [ ": "[", " ] ": "]",
        "df  ": "df_", " zeile ": " head ", " spalte ": " col "
    }
    for falsch, richtig in syntax_fixes.items():
        resultat = resultat.replace(falsch, richtig)
    # GLOBAL_MODEL Kontext für Debugging
    print(f"🧠 NLP-EXTRACT ({GLOBAL_MODEL}): '{resultat[:80]}...'")
    return resultat.strip()

def code_integrity_guard(nlp_pre_cleaned_text: str, evidence: list) -> tuple[str, bool]:
    """
    Zuständigkeit: TECHNISCHE FINALISIERUNG V5 (Syntax Integrity).
    - Heilt Tippfehler/Namen basierend auf RADAR (Universal-Radar v3)
    - SCHÜTZT Python-Syntax 100%
    - GLOBAL_MODEL + PROJECT_PATHS-kompatibel
    """
    try:
        if evidence is None:
            evidence = []

        sanitized = re.sub(r'\s+', ' ', nlp_pre_cleaned_text.strip())

        # 1. PRÄFIX-FIX (ximbond → x inbound)
        sanitized = re.sub(r'\b([xy])([a-zA-Z_]\w*)', r'\1 \2', sanitized, flags=re.IGNORECASE)

        # 2. DE→TECH MAPPING (Space-Padding für Tokenisierung)
        tech_mapping = {
            r'\b(mb|kb|gb)\b': 'memory_usage_mb', r'\bspeicher\b': 'memory_usage_mb',
            r'\b(an(zahl|fang)|count|wieviele)\b': 'row_count', r'\bgröße\b': 'shape_info',
            r'\bdiag(ramm?|plot)\b': 'plot', r'\b(balken|bar)\b': 'bar_chart',
            r'\b(linie|line)\b': 'line_chart', r'\b(punkt|scatter)\b': 'scatter_plot'
        }
        for pattern, replacement in tech_mapping.items():
            sanitized = re.sub(pattern, f' {replacement} ', sanitized, flags=re.IGNORECASE)

        # 3. RADAR-BEST-DATAFRAME (Spalten-reichste Übereinstimmung)
        df_evidenz = [e for e in evidence if e.get('typ') == 'DATAFRAME']
        best_df = None
        if df_evidenz:
            best_df = max(df_evidenz, key=lambda x: len(x.get('identifizierte_spalten', [])))

        # 4. DATAFRAME-HEILUNG (df → df_sample)
        df_placeholder = re.findall(r'\b(df|cl|data)\w*\b', sanitized.lower())
        for placeholder in set(df_placeholder):
            if best_df:
                target_df = best_df['objekt']
                sanitized = re.sub(rf'\b{placeholder}\b', f' {target_df} ', sanitized, flags=re.IGNORECASE)

        # 5. SPALTEN-TYPO-FIX (Radar-Korrekturen)
        for e in df_evidenz:
            if 'korrektur_map' in e:
                for falsch, richtig in e['korrektur_map'].items():
                    pattern = re.compile(rf'\b{re.escape(falsch)}\b', re.IGNORECASE)
                    sanitized = pattern.sub(f' {richtig} ', sanitized)

        # 6. SYNTAX-PROTECTION (Python-only Zeichen)
        allowed_chars = r'[\w\s\.\(\)\[\]\{\}\:\;\,\=\+\-\*\/\\%\&\|\^\~\<\>"]'
        sanitized = re.sub(f'[^{allowed_chars}]', ' ', sanitized)
        sanitized = re.sub(r'\s+', ' ', sanitized).strip()

        # 7. MINIMAL VALIDATION (Python-Code? → Accept)
        print(f"🔍 DEBUG GUARD: len='{len(sanitized)}' | content='{sanitized}'")
        is_python_valid = len(sanitized) > 2  # TEMP: Nur Länge prüfen

        print(f"✅ GUARD V5 ({GLOBAL_MODEL}): '{sanitized[:70]}...'")
        print(f"🔍 FINAL DEBUG: '{sanitized}' → len={len(sanitized)} → valid={is_python_valid}")
        return sanitized, is_python_valid

    except Exception as e:
        print(f"💥 GUARD_CRASH ({GLOBAL_MODEL}): {str(e)[:50]}")
        return nlp_pre_cleaned_text.strip(), False

In [6]:
# === AGENTEN-DEPLOYMENT SYSTEM ===
def deploy_agent_text_datascientist(setup_data: dict) -> str:
    """
    DNA: EVIDENZ-NOTAR V1 (Technical Evidence Notary)
    Erstellt YAML-Agenten-Profil aus __LLM_GLOBAL__ Setup.
    Deployment-Pfad: BEHAVIOR_DIR/text_datascientist.yaml
    """
    config = setup_data["config"]

    agent_config = {
        "agent_name": "TEXT_DATASCIENTIST",
        "role": "Senior Data Scientist | Evidence Notary",
        "model": setup_data.get("model_name", GLOBAL_MODEL),
        "global_model": GLOBAL_MODEL,
        "exit_signal": config["exit_signal"],
        "version": setup_data.get("version", "v1"),

        "instructions": (
            f"DU BIST TECHNISCHER NOTAR | REIN EVIDENZBASIERT\n"
            f"PRIORITÄT: {config['context_priority'].upper()}\n"
            f"{config['instruction_suffix']}\n\n"
            f"RULES:\n"
            f"• df_name = IMMER ORIGINAL\n"
            f"• cl_name = Korrekturen NUR\n"
            f"• v_name = cl.combine_first(df)\n"
            f"• Code: EN | Info: DE"
        ),

        "settings": {
            "temperature": 0.0,
            "num_ctx": 4096,  # Agenten-spezifisch
            "seed": config["seed"],
            "num_thread": config.get("num_thread", 4)
        },

        "tools": ["Pandas_Core", "AutoViz", "Vader_Sentiment"]  # Registry-Referenz
    }

    # Deployment in BEHAVIOR_DIR (PROJECT_PATHS-kompatibel)
    file_path = os.path.join(BEHAVIOR_DIR, "text_datascientist.yaml")

    try:
        os.makedirs(BEHAVIOR_DIR, exist_ok=True)
        with open(file_path, "w", encoding="utf-8") as f:
            yaml.dump(agent_config, f, default_flow_style=False, allow_unicode=True)

        print(f"✅ Agent 'TEXT_DATASCIENTIST' deployed: {file_path}")
        return file_path

    except Exception as e:
        print(f"⚠️ DEPLOY_ERROR: {str(e)}")
        return None

def deploy_agents_yaml() -> bool:
    """
    Zentrale Agenten-Initialisierung.
    - Erstellt alle YAML-Agenten aus __LLM_GLOBAL__ Setup
    - Verwendet BEHAVIOR_DIR (PROJECT_PATHS)
    - Return: True wenn erfolgreich
    """
    try:
        # Zentrale Setup (GLOBAL_MODEL + Config)
        setup_data = __LLM_GLOBAL__()

        # Deploy spezifische Agenten
        agent_file = deploy_agent_text_datascientist(setup_data)

        if agent_file and os.path.exists(agent_file):
            print(f"✅ Agents deployed → BEHAVIOR_DIR/{os.path.basename(agent_file)}")
            print(f"   LLM: {GLOBAL_MODEL} | Tools: {len(setup_data.get('tools_status', {}))} aktiv")
            return True
        else:
            print("❌ Deployment fehlgeschlagen: Keine YAML-Datei erstellt")
            return False

    except Exception as e:
        print(f"💥 AGENTS_DEPLOY_ERROR: {str(e)}")
        return False


# === INITIAL AGENT DEPLOYMENT ===
if __name__ == "__main__":
    success = deploy_agents_yaml()
    if success:
        print("🚀 Agenten-System bereit | GLOBAL_MODEL:", GLOBAL_MODEL)

✅ Tool-Registry erstellt: 17 aktive Tools
✅ Agent 'TEXT_DATASCIENTIST' deployed: /Users/cristallagus/Desktop/GitHub/Projecten/🎭Kundenstimung auf Twitter Verstehen/registry/agent/text_datascientist.yaml
✅ Agents deployed → BEHAVIOR_DIR/text_datascientist.yaml
   LLM: deepseek-coder-v2 | Tools: 17 aktiv
🚀 Agenten-System bereit | GLOBAL_MODEL: deepseek-coder-v2


In [ ]:
# schnelle Frage veredeler
def frage_schnell(unsaubere_eingabe: str, LLM_GLOBAL=None, language: str = 'DE') -> str:
    """
    PURER VEREDLER (v4 - LIGHTNING Edition)
    MAXIMALE GESCHWINDIGKEIT | MINIMALER OUTPUT
    - Kein Spinner | Kein Deep-Audit | Direkt zur Pipeline
    """
    print(f"⚡ [SCHNELL] '{unsaubere_eingabe}' → ", end="", flush=True)

    # 0. Ollama-Check (silent)
    if not pruefe_und_starte_ollama():
        return "🛑 OLLAMA OFFLINE"

    # 1. Setup (minimal)
    if LLM_GLOBAL is None:
        LLM_GLOBAL = __LLM_GLOBAL__()

    try:
        # RADAR (ultra-fast)
        evidenz = lokalisiere_alle_quellen_final(unsaubere_eingabe, LLM_GLOBAL)

        # TOOL-STATUS (cached)
        tool_status = check_available_tools()

        # VEREDLUNG (minimal)
        strenge = apply_system_discipline(language)
        dossier_text, _ = get_system_logic_dossier(tool_status)
        prompt = build_veredelungs_prompt(unsaubere_eingabe, evidenz, dossier_text, strenge)

        # LLM-CALL
        payload = build_llm_payload(prompt, LLM_GLOBAL)
        roh_antwort = execute_api_raw(payload, LLM_GLOBAL["config"])

        # CLEAN + GUARD (1-Pass)
        bereinigte_query = apply_radical_cleaning(roh_antwort, LLM_GLOBAL["config"]["exit_signal"])
        user_query, ist_valide = code_integrity_guard(bereinigte_query, evidenz)

        if not ist_valide:
            print("❌ INVALID")
            return "❌ UNGÜLTIG"

        # PIPELINE (instant)
        pipeline = erstelle_sequenz_aus_registry(user_query, tool_status)
        pipeline_str = " → ".join([s['name'] for s in pipeline[:3]])  # Max 3

        # ULTRA-KOMPAKT OUTPUT
        status = "✅" if pipeline else "➡️"
        print(f"{status} '{user_query}' | {pipeline_str}")
        return user_query

    except Exception as e:
        print(f"💥 {str(e)[:30]}")
        return f"ERROR: {str(e)}"

In [7]:
# Intensive Frage veredler
def frage(unsaubere_eingabe: str, LLM_GLOBAL=None, language: str = 'DE') -> str:
    """
    PURER VEREDLER V4 (Think-Traceability Edition)
    - Deep-Audit + RADAR + LLM + Guard + Pipeline
    - Think-Variante für transparente Entscheidungen
    """
    # Audit-Trail initialisieren
    audit_trail = {
        "input": unsaubere_eingabe,
        "quellen_check": "FAILED",
        "llm_entscheidung": "WAITING",
        "guard_aktion": "NONE",
        "pipeline": "NONE"
    }
    print(f"\n🔍 [AUDIT 0] EINGANG: '{unsaubere_eingabe}'")
    print(f"   LLM: {GLOBAL_MODEL}")

    # Ollama-Check
    if not pruefe_und_starte_ollama():
        return "🛑 ABBRUCH: Ollama offline."
    # Zentrale Setup
    if LLM_GLOBAL is None:
        LLM_GLOBAL = __LLM_GLOBAL__()
    # Think-Spinner
    stop_event = threading.Event()
    spinner_thread = threading.Thread(target=start_spinner, args=(stop_event, language.upper() == 'DE'))
    spinner_thread.start()
    try:
        # === THINK-STEP 1: RADAR (Quellen) ===
        evidenz = lokalisiere_alle_quellen_final(unsaubere_eingabe, LLM_GLOBAL)
        if evidenz:
            audit_trail["quellen_check"] = "SUCCESS"
            for e in evidenz[:2]:  # Top 2
                spalten = e.get('identifizierte_spalten', [])
                print(f"📍 [RADAR] {e['objekt']} | Spalten: {spalten[:3]}")
        else:
            print("📍 [RADAR] Keine DataFrames/Spalten erkannt")
        # === THINK-STEP 2: TOOL-INVENTAR ===
        tool_status = check_available_tools()
        aktive_tools = [t for t, info in tool_status.items() if info.get('phase')]
        print(f"🛠️  Tools verfügbar: {len(aktive_tools)}")
        # === THINK-STEP 3: VEREDLUNG ===
        strenge = apply_system_discipline(language)
        dossier_text, _ = get_system_logic_dossier(tool_status)
        prompt = build_veredelungs_prompt(unsaubere_eingabe, evidenz, dossier_text, strenge)
        payload = build_llm_payload(prompt, LLM_GLOBAL)
        roh_antwort = execute_api_raw(payload, LLM_GLOBAL["config"])
        audit_trail["llm_entscheidung"] = roh_antwort[:100]
        print(f"🧠 [LLM] Roh: '{roh_antwort[:40]}...'")
        # === THINK-STEP 4: CLEAN + GUARD ===
        bereinigte_query = apply_radical_cleaning(roh_antwort, LLM_GLOBAL["config"]["exit_signal"])
        user_query, ist_valide = code_integrity_guard(bereinigte_query, evidenz)
        audit_trail["guard_aktion"] = f"'{user_query}' (valid: {ist_valide})"
        if not ist_valide:
            print(f"❌ [GUARD] REJECTED: '{user_query}'")
            return "❌ Ungültiger Befehl nach Verarbeitung"
        # === THINK-STEP 5: PIPELINE ===
        pipeline = erstelle_sequenz_aus_registry(user_query, tool_status)
        pipeline_names = [step['name'] for step in pipeline]
        audit_trail["pipeline"] = " → ".join(pipeline_names)
        # === THINK-STEP 6: EXECUTION-READY OUTPUT ===
        print("\n" + "="*60)
        print("🤔 THINK-TRACE (V4):")
        print(f"  💡 INPUT     → '{unsaubere_eingabe}'")
        print(f"  📍 QUELLE    → {audit_trail['quellen_check']} ({len(evidenz) if evidenz else 0} hits)")
        print(f"  🧠 LLM       → '{audit_trail['llm_entscheidung'][:30]}...'")
        print(f"  🛡️  GUARD    → {audit_trail['guard_aktion']}")
        print(f"  🔗 PIPELINE  → {audit_trail['pipeline'] or 'Direkt-Exec'}")
        print("="*60)
        print(f"✅ EXECUTE: {user_query}")
        return user_query

    except Exception as e:
        print(f"💥 [CRASH] {str(e)}")
        return f"ERROR: {str(e)}"

    finally:
        stop_event.set()
        spinner_thread.join()
        print()

In [8]:
frage("zeige mir die ersten 5 zeilen von df")


🔍 [AUDIT 0] EINGANG: 'zeige mir die ersten 5 zeilen von df'
   LLM: deepseek-coder-v2
✅ Multitalent-Check: 'deepseek-coder-v2' ist einsatzbereit.
✅ Tool-Registry erstellt: 17 aktive Tools
🤖 LLM denkt... (GPU/CPU) ⠋📍 [RADAR] df_sample | Spalten: []
✅ Tool-Registry erstellt: 17 aktive Tools
🛠️  Tools verfügbar: 16
🤖 LLM denkt... (GPU/CPU) ⠏ RAM↓0.7GB|GC🧠 [LLM] Roh: 'df_sample.head(5)...'
🧹 RADICAL-CLEAN V6: 'df_sample.head(5)...'
🔍 DEBUG GUARD: len='17' | content='df_sample.head(5)'
✅ GUARD V5 (deepseek-coder-v2): 'df_sample.head(5)...'
🔍 FINAL DEBUG: 'df_sample.head(5)' → len=17 → valid=True

🤔 THINK-TRACE (V4):
  💡 INPUT     → 'zeige mir die ersten 5 zeilen von df'
  📍 QUELLE    → SUCCESS (1 hits)
  🧠 LLM       → 'df_sample.head(5)...'
  🛡️  GUARD    → 'df_sample.head(5)' (valid: True)
  🔗 PIPELINE  → Direkt-Exec
✅ EXECUTE: df_sample.head(5)
                                                                                                    


'df_sample.head(5)'

In [9]:
frage("mach mir ein verticaler PLot das die df_sam die spalte imbond ")


🔍 [AUDIT 0] EINGANG: 'mach mir ein verticaler PLot das die df_sam die spalte imbond '
   LLM: deepseek-coder-v2
✅ Multitalent-Check: 'deepseek-coder-v2' ist einsatzbereit.
✅ Tool-Registry erstellt: 17 aktive Tools
🤖 LLM denkt... (GPU/CPU) ⠋📍 [RADAR] df_sample | Spalten: ['inbound']
💥 [CRASH] 'objekt'
                                                                                                    


"ERROR: 'objekt'"

In [ ]:
frage('mit pandasai Als Nächstes auf das visuelle Verständnis der Daten. Mit AutoViz können Diagramme und Grafiken erstellen, die wichtige Erkenntnisse über die Daten liefern. Vergleichen die Ergebnisse von AutoViz mit den vorherigen Vorschlägen von LLM.')

In [ ]:
frage('wie fiel MB hatt der df_sample')

In [ ]:
frage('erstelle mir ein Balken diagran das die menge der tweat anzeigt als count beschrifte alle axen')

# Integritäts test

In [ ]:
# 1. LLM VERBINDUNGS-CHECK (Auto-Check/Start)
def pruefe_und_starte_ollama():
    url = "http://localhost:11434"
    try:
        if requests.get(url, timeout=1).status_code == 200: return True
    except: pass
    try:
        if sys.platform == "darwin":
            subprocess.Popen(["open", "-a", "Ollama"])
        elif sys.platform == "win32":
            p = os.path.expandvars(r"%LocalAppData%\Ollama\ollama app.exe")
            subprocess.Popen([p] if os.path.exists(p) else ["ollama", "serve"])
            if os.path.exists(p):
                subprocess.Popen([p])
            else:
                subprocess.Popen(["ollama", "serve"], shell=True)
        for _ in range(10):
            time.sleep(1)
            try:
                if requests.get(url, timeout=1).status_code == 200:
                    return True
            except: continue
    except Exception as e:
        print(f"⚠️ Fehler beim Startversuch: {e}")
    print("\n🛑 START FEHLGESCHLAGEN. Nächste Schritte:")
    print("1. Installieren (ollama.com)")
    return
pruefe_und_starte_ollama()

In [ ]:
# Check der Importierten Tools
display(check_available_tools())

In [ ]:
def test_system_integrity_full_audit(LLM_GLOBAL=None):
    """
    Zuständigkeit: Unit-Test für den Universal-Radar (V25).
    Features:
    - Prüft RAM (df) & Schablone (cl).
    - Validiert Web-Live & Filtert tote Links.
    - Projekt-Validierung: Prüft Erreichbarkeit der DIST-Ordner (output/).
    """
    import gc, os
    if LLM_GLOBAL is None:
        LLM_GLOBAL = __LLM_GLOBAL__()

    print(f"\n🔍 STARTE SYSTEM-RADAR AUDIT (V25 - PROJECT-AWARE)")
    print("-" * 70)

    # --- 1. TEST-OBJEKTE REGISTRIEREN (GLOBAL) ---
    globals()['df_audit_ram'] = pd.DataFrame({'status': ['OK']})
    globals()['cl_audit_layer'] = pd.DataFrame({'fix': [1]}) # Schablonen-Test

    globals()['df_audit_web_ok'] = pd.DataFrame({'ping': [1]})
    globals()['df_audit_web_ok'].attrs['source'] = "https://www.google.com"

    globals()['df_audit_web_dead'] = pd.DataFrame({'ping': [0]})
    globals()['df_audit_web_dead'].attrs['source'] = "https://dieser-link-ist-tot-999.de"

    # NEU: Wir nutzen deinen Projekt-Ordner statt Systempfade (Disk-Fix)
    test_file_name = "audit_disk_test.txt"
    test_path = os.path.join(PROJECT_PATHS["DIST_TEXT"], test_file_name)
    with open(test_path, 'w', encoding='utf-8') as f: f.write("Radar Test")

    # --- 2. EVALUIERUNG ---

    # Test A: RAM & Schablone
    res_ram = lokalisiere_alle_quellen_final("df_audit_ram cl_audit_layer", LLM_GLOBAL)
    success_ram = any(d.get('objekt') == 'df_audit_ram' for d in res_ram) if res_ram else False
    success_cl = any(d.get('status') == 'ONLINE (RAM_CLEAN_LAYER)' for d in res_ram) if res_ram else False
    print(f"{'✅' if success_ram and success_cl else '❌'} RAM & SCHABLONE: Erkannt")

    # Test B: Web-Live (Google)
    res_web = lokalisiere_alle_quellen_final("df_audit_web_ok", LLM_GLOBAL)
    success_web = any("ONLINE (WEB)" in str(d) for d in res_web) if res_web else False
    print(f"{'✅' if success_web else '❌'} WEB-CONNECT: Google Online")

    # Test C: Filter (Tote Quelle)
    res_dead = lokalisiere_alle_quellen_final("df_audit_web_dead", LLM_GLOBAL)
    success_dead = not any(d.get('objekt') == 'df_audit_web_dead' for d in res_dead) if res_dead else True
    print(f"{'✅' if success_dead else '❌'} FILTER-LOGIK: Tote Links blockiert")

    # Test D: Disk-Pfad (Projekt-Integrität)
    # Wir suchen nach der gerade erstellten Test-Datei
    res_disk = lokalisiere_alle_quellen_final("audit_disk_test", LLM_GLOBAL)
    success_disk = any(d.get('typ') == 'OUTPUT_QUELLE' for d in res_disk) if res_disk else False
    print(f"{'✅' if success_disk else '❌'} DISK-VALIDIERUNG: Projekt-Ordner (output/) erkannt")

    # 3. CLEANUP (RAM-Schonung & Dateisystem)
    if os.path.exists(test_path):
        os.remove(test_path)

    for obj in ['df_audit_ram', 'cl_audit_layer', 'df_audit_web_ok', 'df_audit_web_dead']:
        if obj in globals(): del globals()[obj]

    gc.collect()

    overall = all([success_ram, success_cl, success_web, success_dead, success_disk])
    print("-" * 70)
    print(f"ERGEBNIS: {'INTEGRITÄT GEWÄHRLEISTET' if overall else 'FEHLER GEFUNDEN'}")
    return overall

# Audit ausführen
test_system_integrity_full_audit()